In [66]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWHUSKY'
SS_VER = 'SS_VER_2_1'

In [67]:
%run "../Setup_Scripts/Setup_Generic.ipynb"

INFO: Caught exception on reconnecting to target - attempting to reconnect to scope first.
INFO: This is a work-around when USB has died without Python knowing. Ignore errors above this line.
INFO: Found ChipWhisperer😍
scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 346246                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1     

In [60]:
import subprocess
from IPython.display import HTML, display
import html

cmd = f"""
cd ../../firmware/mcu/simpleserial-xor
make PLATFORM={PLATFORM} SS_VER={SS_VER} CRYPTO_TARGET=MBEDTLS CRYPTO_OPTIONS=AES128C -j
"""

result = subprocess.run( cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

display(HTML(f"""
<div style="max-height: 300px; overflow-y: auto; border: 1px solid #888; padding: 8px; background-color: #111; color: #ddd; font-family: monospace;
    white-space: pre-wrap;">
{html.escape(result.stdout)}
</div>
"""))

In [61]:
cw.program_target(scope, prog, "../../firmware/mcu/simpleserial-xor/simpleserial-xor-{}.hex".format(PLATFORM))

In [68]:
def hexstr(data):
    if data is None:
        return "None"
    return " ".join(f"{b:02X}" for b in data)


In [69]:
# simple xor

ktp = cw.ktp.Basic()
key, text = ktp.next()

target.flush()

target.simpleserial_write('b', key)
ack = target.simpleserial_wait_ack()

target.simpleserial_write('a', text)

response = target.simpleserial_read('r', 16)

expected = bytearray(t ^ k for t, k in zip(text, key))

print("received:", hexstr(response))
print("expected:", hexstr(expected))
print("match   :", response == expected)

received: AA 25 CB BE 28 88 7B 90 2C 6A D6 70 82 8E D3 B8
expected: AA 25 CB BE 28 88 7B 90 2C 6A D6 70 82 8E D3 B8
match   : True


In [77]:
import chipwhisperer as cw
from Crypto.Cipher import AES

# Generate key and plaintext
ktp = cw.ktp.Basic()
key, _ = ktp.next()

# Initial CTR counter
counter = bytearray([0x00] * 16)

plaintext = bytearray(range(1, 33))

target.flush()

# Set AES key
target.simpleserial_write('b', key)
ack = target.simpleserial_wait_ack()

# Set counter 
target.simpleserial_write('e', counter)
ack = target.simpleserial_wait_ack()

# Encrypt plaintext on SAM4S
target.simpleserial_write('c', plaintext)

ciphertext = target.simpleserial_read('r', 32)

# Decrypt plaintext on SAM4S
target.simpleserial_write('d', ciphertext)

decrypted = target.simpleserial_read('r', 32)

# --------------------------------------------------
# Calculate expected AES-128-CTR on PC
# --------------------------------------------------

aes = AES.new(bytes(key), AES.MODE_ECB)

expected = bytearray()
keystream = bytearray()

counter_value = int.from_bytes(counter, byteorder="big")

for offset in range(0, len(plaintext), 16):

    # Current 128-bit counter
    ctr = counter_value.to_bytes(16, byteorder="big")

    # AES(K, counter)
    ks = aes.encrypt(ctr)
    keystream.extend(ks)

    # Remaining plaintext, max. 16 bytes
    block = plaintext[offset:offset + 16]

    # XOR plaintext with keystream
    expected.extend(
        p ^ k
        for p, k in zip(block, ks)
    )

    # Increment 128-bit counter
    counter_value = (counter_value + 1) & ((1 << 128) - 1)



print("key        :", hexstr(key))
print("counter    :", hexstr(counter))
print("plaintext  :", hexstr(plaintext))
print("decrypted  :", hexstr(decrypted))
print("keystream  :", hexstr(keystream))
print("ciphertext :", hexstr(ciphertext))
print("expected   :", hexstr(expected))
print("match enc  :", ciphertext == expected)
print("match dec  :", plaintext == decrypted)

key        : 2B 7E 15 16 28 AE D2 A6 AB F7 15 88 09 CF 4F 3C
counter    : 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00
plaintext  : 01 02 03 04 05 06 07 08 09 0A 0B 0C 0D 0E 0F 10 11 12 13 14 15 16 17 18 19 1A 1B 1C 1D 1E 1F 20
decrypted  : 01 02 03 04 05 06 07 08 09 0A 0B 0C 0D 0E 0F 10 11 12 13 14 15 16 17 18 19 1A 1B 1C 1D 1E 1F 20
keystream  : 7D F7 6B 0C 1A B8 99 B3 3E 42 F0 47 B9 1B 54 6F 57 12 7D 40 34 B1 BE BF AE F4 66 B9 C7 72 6F C6
ciphertext : 7C F5 68 08 1F BE 9E BB 37 48 FB 4B B4 15 5B 7F 46 00 6E 54 21 A7 A9 A7 B7 EE 7D A5 DA 6C 70 E6
expected   : 7C F5 68 08 1F BE 9E BB 37 48 FB 4B B4 15 5B 7F 46 00 6E 54 21 A7 A9 A7 B7 EE 7D A5 DA 6C 70 E6
match enc  : True
match dec  : True
